# Reference-Metric Applications

Reference metrics connect coordinate maps to the symbolic and generated infrastructure used by curvilinear NRPy calculations. A `ReferenceMetric` stores the numerical coordinates, the map to Cartesian coordinates, orthogonal scale factors, metric quantities, singular directions, and precompute-ready coordinate factors.

## Table of Contents

1. [Runtime context and imports](#Runtime-context-and-imports)
2. [Coordinate maps and scale factors](#Coordinate-maps-and-scale-factors)
3. [Metric determinant check](#Metric-determinant-check)
4. [Precompute-ready factors](#Precompute-ready-factors)
5. [Next notebooks](#Next-notebooks)

## Runtime Context and Imports

The import location is printed only as runtime context. The notebook uses the importable `nrpy` package and current reference-metric APIs.

In [ ]:
import nrpy
import nrpy.params as par
import nrpy.reference_metric as refmetric
import sympy as sp
from nrpy.infrastructures.BHaH.rfm_precompute import ReferenceMetricPrecompute


print("Imported nrpy from:", nrpy.__file__)

## Coordinate Maps and Scale Factors

The same interface covers Cartesian, spherical, and stretched curvilinear coordinates. The compact summaries below show the numerical coordinates, Cartesian map, orthogonal scale factors, metric diagonals, singular directions, and representative Jacobian entries.

In [ ]:
def short(expr, width=88):
    text = sp.sstr(expr)
    return text if len(text) <= width else text[: width - 3] + "..."


def short_list(values):
    return [short(value) for value in values]


def metric_summary(coord_system):
    metric = refmetric.ReferenceMetric(coord_system, enable_rfm_precompute=False)
    print(f"\n{coord_system}")
    print("  xx:", short_list(metric.xx))
    print("  xx_to_Cart:", short_list(metric.xx_to_Cart))
    print("  scale factors:", short_list(metric.scalefactor_orthog))
    print("  ghatDD diagonal:", short_list(metric.ghatDD[i][i] for i in range(3)))
    print("  ghatUU diagonal:", short_list(metric.ghatUU[i][i] for i in range(3)))
    print("  detgammahat:", short(metric.detgammahat))
    print("  radial-like directions:", metric.radial_like_dirns)
    print(
        "  Jacobian samples:",
        short_list(
            [
                metric.Jac_dUCart_dDrfmUD[0][0],
                metric.Jac_dUCart_dDrfmUD[0][1],
                metric.Jac_dUCart_dDrfmUD[1][0],
                metric.Jac_dUCart_dDrfmUD[2][2],
            ]
        ),
    )
    return metric


metrics = {
    coord_system: metric_summary(coord_system)
    for coord_system in [
        "Cartesian",
        "Spherical",
        "SinhSpherical",
        "SinhCylindrical",
    ]
}

## Metric Determinant Check

For an orthogonal reference metric, the determinant is the square of the product of the scale factors. The residual below checks that identity for spherical coordinates.

In [ ]:
spherical = metrics["Spherical"]
scale_product = sp.prod(spherical.scalefactor_orthog)
determinant_residual = sp.trigsimp(
    sp.simplify(spherical.detgammahat - scale_product**2)
)
print("Spherical determinant residual:", determinant_residual)

## Precompute-Ready Factors

Stretched coordinate systems contain repeated functions of individual numerical coordinates. RFM precompute extracts those factors so generated code can allocate, fill, and reuse them.

In [ ]:
par.set_parval_from_str("parallelization", "openmp")
par.set_parval_from_str("fp_type", "double")
par.set_parval_from_str(
    "CoordSystem_to_register_CodeParameters", "SinhCylindrical"
)

precompute_metric = refmetric.reference_metric["SinhCylindrical_rfm_precompute"]
precompute = ReferenceMetricPrecompute("SinhCylindrical")

print("Precompute metric:", precompute_metric.CoordSystem)
print("Stored member count:", len(precompute.member_specs))
print("First member names:", [name for name, _ in precompute.member_specs[:6]])

## Next Notebooks

[Basis transforms](basis_transforms.ipynb) applies the Jacobians stored on reference metrics. [Curvilinear wave equation](../3-wave_equation/wave_equation_curvilinear.ipynb) uses reference metrics and precompute-ready factors in a generated wave-equation workflow.